In [1]:
import os
import sys
from pathlib import Path

library_path = os.path.abspath('../src')
if library_path not in sys.path:
    sys.path.append(library_path)
library_path = Path(library_path)

# load data
DATA_PATH = library_path.parent / "data"
PLOTS_PATH = library_path.parent / "plots"

In [2]:
import pandas as pd

In [4]:
df = pd.read_csv(DATA_PATH / "processed_data.csv", sep="\t")
df.head()

,dataset,MO5L,SC5L,UA5L,PD5L,AD5L,EQvas,srh,sat,mo2cat,...,educ_pst,smoke_ever,smoke_ecig,alcohol_yr,diabetes,obese,resp,bowel,mus,skin
0,HSE 2017,No Problems,No Problems,No Problems,No Problems,No Problems,100.0,Very good/Excellent,9.0,No Problems,...,NaN,yes,no,monthly+,0.0,NaN,0.0,NaN,NaN,0.0
1,HSE 2017,Slight,No Problems,No Problems,Moderate,Slight,75.0,Good/fair,7.0,Any Problems,...,NaN,NaN,yes,weekly+,0.0,0.0,0.0,NaN,NaN,0.0
2,HSE 2017,No Problems,No Problems,No Problems,No Problems,No Problems,90.0,Very good/Good,9.0,No Problems,...,NaN,NaN,no,weekly+,0.0,1.0,0.0,NaN,NaN,0.0
3,HSE 2017,Moderate,Slight,Moderate,Moderate,Slight,70.0,Good/fair,9.0,Any Problems,...,NaN,no,no,monthly+,0.0,1.0,0.0,NaN,NaN,0.0
4,HSE 2017,No Problems,No Problems,No Problems,Slight,No Problems,70.0,Very good/Good,5.0,No Problems,...,NaN,yes,no,monthly+,1.0,1.0,0.0,NaN,NaN,0.0


In [6]:
hse_2017 = df['dataset'] == "HSE 2017"
hse_2018 = df['dataset'] == "HSE 2018"
daph_2023 = df['dataset'] == "DAPHNIE 2023"
daph_2024 = df['dataset'] == "DAPHNIE 2024"
df_reduc =  df[hse_2017 | hse_2018 | daph_2023 | daph_2024].copy()
print(f"Reduced dataset to HSE 2017-2018 and DAPHNIE 2023-2024. Remaining rows: {len(df_reduc):,} of {len(df):,}")

Reduced dataset to HSE 2017-2018 and DAPHNIE 2023-2024. Remaining rows: 33,208 of 49,141


In [7]:
df_reduc['type'] = df_reduc['dataset'].apply(lambda x: 'HSE' if 'HSE' in x else 'DAPHNIE')

## Covariate Shift Framing
We treat `HSE` versus `DAPHNIE` as a domain-classification problem using background covariates only.

The downstream targets `LSS_rs`, `EQ_index`, and `EQvas`, along with EQ-5D profile variables and close derivatives, are excluded from this stage to avoid leakage.

If the two domains are separable under these covariates, that supports a covariate-shift interpretation and gives us estimated density-ratio weights for downstream transfer.

In [8]:
downstream_targets = ["LSS_rs", "EQ_index", "EQvas"]
print("Downstream targets reserved for later transfer models:", ", ".join(downstream_targets))

Downstream targets reserved for later transfer models: LSS_rs, EQ_index, EQvas


In [11]:
import numpy as np
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score
from sklearn.model_selection import StratifiedKFold, cross_val_predict
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder

domain_covariates = [
    "age7cat",
    "Sex",
    "eth4cat",
    "emp_cat",
    "edu_cat",
    "educ_pst",
    "smoke_ever",
    "smoke_ecig",
    "alcohol_yr",
    "diabetes",
    "obese",
    "resp",
    "bowel",
    "mus",
    "skin",
]

missing_covariates = [column for column in domain_covariates if column not in df_reduc.columns]
if missing_covariates:
    raise ValueError(f"Missing expected covariates: {missing_covariates}")

df_domain = df_reduc.loc[:, ["dataset", "type", *domain_covariates]].copy()
df_domain[domain_covariates] = df_domain[domain_covariates].astype("object")
df_domain["is_daphnie"] = (df_domain["type"] == "DAPHNIE").astype(int)

print(f"Modeling rows: {len(df_domain):,}")
print(df_domain["type"].value_counts(dropna=False).rename("count"))

Modeling rows: 33,208
type
DAPHNIE    17033
HSE        16175
Name: count, dtype: int64


In [12]:
X_domain = df_domain[domain_covariates]
y_domain = df_domain["is_daphnie"]

domain_pipeline = Pipeline(
    steps=[
        (
            "categorical",
            ColumnTransformer(
                transformers=[
                    (
                        "all_categorical",
                        Pipeline(
                            steps=[
                                ("imputer", SimpleImputer(strategy="most_frequent")),
                                ("encoder", OneHotEncoder(handle_unknown="ignore", sparse_output=True)),
                            ]
                        ),
                        domain_covariates,
                    )
                ]
            ),
        ),
        ("classifier", LogisticRegression(max_iter=1000, class_weight="balanced")),
    ]
)

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
domain_prob_cv = cross_val_predict(
    domain_pipeline,
    X_domain,
    y_domain,
    cv=cv,
    method="predict_proba",
    n_jobs=-1,
 )[:, 1]

domain_auc = roc_auc_score(y_domain, domain_prob_cv)
print(f"Cross-validated ROC AUC for HSE vs DAPHNIE: {domain_auc:.3f}")

df_domain["p_daphnie_given_x"] = np.clip(domain_prob_cv, 1e-3, 1 - 1e-3)
df_domain["p_hse_given_x"] = 1 - df_domain["p_daphnie_given_x"]

# Density-ratio weights for transporting HSE observations toward the DAPHNIE covariate distribution.
df_domain["w_hse_to_daphnie"] = df_domain["p_daphnie_given_x"] / df_domain["p_hse_given_x"]
df_domain["w_hse_to_daphnie_clipped"] = df_domain["w_hse_to_daphnie"].clip(
    upper=df_domain.loc[df_domain["type"] == "HSE", "w_hse_to_daphnie"].quantile(0.99)
 )

df_domain[["type", "p_daphnie_given_x", "w_hse_to_daphnie", "w_hse_to_daphnie_clipped"]].head()

Cross-validated ROC AUC for HSE vs DAPHNIE: 0.997


,type,p_daphnie_given_x,w_hse_to_daphnie,w_hse_to_daphnie_clipped
0,HSE,0.001000,0.001001,0.001001
1,HSE,0.001000,0.001001,0.001001
2,HSE,0.001000,0.001001,0.001001
3,HSE,0.001062,0.001063,0.001063
4,HSE,0.001000,0.001001,0.001001


In [13]:
domain_pipeline.fit(X_domain, y_domain)

feature_names = domain_pipeline.named_steps["categorical"].get_feature_names_out()
coefficients = domain_pipeline.named_steps["classifier"].coef_.ravel()

coef_table = (
    pd.DataFrame({"feature": feature_names, "coefficient": coefficients})
    .assign(abs_coefficient=lambda frame: frame["coefficient"].abs())
    .sort_values("abs_coefficient", ascending=False)
    .reset_index(drop=True)
 )

print("Largest positive coefficients favor DAPHNIE; largest negative coefficients favor HSE.")
coef_table.loc[:, ["feature", "coefficient"]].head(20)

Largest positive coefficients favor DAPHNIE; largest negative coefficients favor HSE.


,feature,coefficient
0,all_categorical__eth4cat_Others,7.673117
1,all_categorical__eth4cat_White,-4.500489
2,all_categorical__edu_cat_No qualification,-2.317926
3,all_categorical__eth4cat_Asian,-2.126253
4,all_categorical__edu_cat_Below degree,1.874103
5,all_categorical__age7cat_6.0,1.586941
6,all_categorical__eth4cat_Black,-1.538739
7,all_categorical__bowel_0.0,-1.514173
8,all_categorical__smoke_ever_yes,-1.486398
9,all_categorical__alcohol_yr_never,1.417576
